In [29]:
# ── Config ────────────────────────────────────────────────────────────────────
# Set the experiment number to load (picks the latest run automatically)
EXPERIMENT = 200
# Which evaluation split to display: 'train' or 'test'
# Both live in the same row — train_* vs test_* column prefixes.
SPLIT = 'test'

In [30]:
import pandas as pd
from pathlib import Path

results_root = Path('..') / 'results'

# Find the latest run folder for the given experiment
prefix = f'experiment_{EXPERIMENT}_'
runs = sorted([d for d in results_root.iterdir() if d.is_dir() and d.name.startswith(prefix)])
assert runs, f'No runs found for experiment {EXPERIMENT}'
run_dir = runs[-1]
print(f'Loading: {run_dir.name}')

df = pd.read_parquet(run_dir / 'process_eval_results.parquet')
# The 'split' column marks the training set used (always 'TRAIN' or 'ALL DATA').
# Train vs test evaluation results are distinguished by column prefix: train_* vs test_*.
# Check the requested prefix exists.
split_prefix = SPLIT.lower() + '_'
available_prefixes = set()
for c in df.columns:
    for p in ('train_', 'test_'):
        if c.startswith(p):
            available_prefixes.add(p.rstrip('_'))
print(f'Available column prefixes: {sorted(available_prefixes)}')
assert split_prefix.rstrip('_') in available_prefixes, \
    f'No columns with prefix "{split_prefix}" found. Available: {sorted(available_prefixes)}'

print(f'Split prefix: {split_prefix} | {len(df)} rows | processes: {sorted(df["process"].unique())}')

Loading: experiment_200_20260623_114057
Available column prefixes: ['test', 'train']
Split prefix: test_ | 36 rows | processes: ['process_1', 'process_2', 'process_3', 'process_4']


In [31]:
# ── Mode display labels (mirrors modelling.py _display_mode) ─────────────────
def display_mode(m):
    m = str(m)
    if not m.startswith('petri_net_'):
        return m
    rest = m[len('petri_net_'):]
    if rest.endswith('_ml_plus_global'):
        return rest[:-len('_ml_plus_global')] + ' / ml_global'
    if rest.endswith('_ml_plus_per_act'):
        return rest[:-len('_ml_plus_per_act')] + ' / ml_local'
    return rest + ' / statistical'

df['mode_label'] = df['mode'].map(display_mode)

In [32]:
# ── Metrics shown in the short heatmap ───────────────────────────────────────
# Matches _plot_short_heatmap: evt count, act duration, edge F1, fitness, precision + overall
METRIC_BASES = [
    'basic_metrics_event_count_error',
    'duration_metrics_activity_duration_error',
    'control_flow_metrics_edge_f1_error',
    'conformance_metrics_fitness_error',
    'conformance_metrics_precision_error',
    'overall_error',
]

split_prefix = SPLIT.lower() + '_'
metric_cols  = [split_prefix + m for m in METRIC_BASES]
metric_cols  = [c for c in metric_cols if c in df.columns]

# Two-line LaTeX column headers using \makecell (requires \usepackage{makecell} in preamble)
METRIC_LABELS = {
    split_prefix + 'basic_metrics_event_count_error':          r'\makecell{Event\\Count Err}',
    split_prefix + 'duration_metrics_activity_duration_error': r'\makecell{Activity\\Dur. Error}',
    split_prefix + 'control_flow_metrics_edge_f1_error':       r'\makecell{Edge\\F1 Error}',
    split_prefix + 'conformance_metrics_fitness_error':        r'\makecell{Fitness\\Error}',
    split_prefix + 'conformance_metrics_precision_error':      r'\makecell{Precision\\Error}',
    split_prefix + 'overall_error':                            r'\makecell{Overall\\Error}',
}

processes = sorted(df['process'].unique())
print(f'Metrics: {metric_cols}')
print(f'Processes: {processes}')
print(f'Modes: {df["mode_label"].unique().tolist()}')

Metrics: ['test_basic_metrics_event_count_error', 'test_duration_metrics_activity_duration_error', 'test_control_flow_metrics_edge_f1_error', 'test_conformance_metrics_fitness_error', 'test_conformance_metrics_precision_error', 'test_overall_error']
Processes: ['process_1', 'process_2', 'process_3', 'process_4']
Modes: ['statistical', 'alpha / statistical', 'heuristic / statistical', 'inductive / statistical', 'combined / statistical', 'combined / ml_global', 'combined / ml_local', 'heuristic / ml_global', 'heuristic / ml_local']


In [33]:
# ── Build table: rows=(process, process model, duration pred), cols=metrics ───
def split_mode(label):
    if ' / ' in label:
        left, right = label.split(' / ', 1)
        return left.strip(), right.strip()
    return label, '—'

overall_col = split_prefix + 'overall_error'

frames = []
for proc in processes:
    sub = df[df['process'] == proc].copy()
    # Sort best (lowest overall error) first
    if overall_col in sub.columns:
        sub = sub.sort_values(overall_col, ascending=True)
    sub = sub.set_index('mode_label')[metric_cols]
    sub.columns = [METRIC_LABELS.get(c, c) for c in sub.columns]
    pmodel, dpred = zip(*[split_mode(m) for m in sub.index])
    sub.index = pd.MultiIndex.from_arrays(
        [[proc] * len(sub), list(pmodel), list(dpred)],
        names=['Process', 'Process Model', 'Duration Pred.']
    )
    frames.append(sub)

table = pd.concat(frames)
table

\makecell{Event\\Count Err}  \
Process   Process Model Duration Pred.                                
process_1 combined      ml_local                           0.000000   
          heuristic     ml_local                           0.000000   
                        ml_global                          0.000000   
          combined      ml_global                          0.000000   
          heuristic     statistical                        0.000000   
          inductive     statistical                        0.000000   
          combined      statistical                        0.000000   
          alpha         statistical                        0.000000   
          statistical   —                                  0.000000   
process_2 combined      ml_local                           0.613636   
          heuristic     ml_local                           0.613636   
                        ml_global                          0.613636   
          combined      ml_global                          0.613636   
                        statistical                        0.613636   
          heuristic     statistical                        0.613636   
          inductive     statistical                        0.192308   
          statistical   —                                  0.641026   
          alpha         statistical                        0.942308   
process_3 heuristic     ml_global                          0.750000   
          combined      ml_global                          0.750000   
                        ml_local                           0.750000   
          heuristic     ml_local                           0.750000   
          combined      statistical                        0.750000   
          heuristic     statistical                        0.750000   
          statistical   —                                  0.777778   
          inductive     statistical                        1.400000   
          alpha         statistical                        0.714286   
process_4 statistical   —                                  0.266055   
          combined      ml_local                           0.440367   
          heuristic     ml_local                           0.440367   
          combined      statistical                        0.440367   
          heuristic     statistical                        0.440367   
          combined      ml_global                          0.440367   
          heuristic     ml_global                          0.440367   
          inductive     statistical                        0.231193   
          alpha         statistical                        0.941284   

                                        \makecell{Activity\\Dur. Error}  \
Process   Process Model Duration Pred.                                    
process_1 combined      ml_local                               0.128949   
          heuristic     ml_local                               0.128949   
                        ml_global                              0.177783   
          combined      ml_global                              0.177783   
          heuristic     statistical                            0.388745   
          inductive     statistical                            0.388745   
          combined      statistical                            0.388745   
          alpha         statistical                            0.388745   
          statistical   —                                      0.404748   
process_2 combined      ml_local                               0.290891   
          heuristic     ml_local                               0.290891   
                        ml_global                              0.370369   
          combined      ml_global                              0.370369   
                        statistical                            0.657237   
          heuristic     statistical                            0.657237   
          inductive     statistical                            0.5

In [36]:
# ── LaTeX output ──────────────────────────────────────────────────────────────
# Requires \usepackage{makecell} and \usepackage{booktabs} in the LaTeX preamble.
import re

latex = table.round(3).to_latex(
    multicolumn=True,
    multicolumn_format='c',
    multirow=True,
    float_format='%.3f',
    escape=False,
    caption=f'Results for experiment {EXPERIMENT} ({SPLIT} evaluation). All metrics: lower is better (0 = best).',
    label=f'tab:exp{EXPERIMENT}_{SPLIT}',
    position='H',
)

# Remove \cline{...} lines — pandas adds them as group separators in multirow tables
latex = re.sub(r'\s*\\cline\{[^}]+\}', '', latex)

print(latex.replace('_', '\_'))

\begin{table}[H]
\caption{Results for experiment 200 (test evaluation). All metrics: lower is better (0 = best).}
\label{tab:exp200\_test}
\begin{tabular}{lllrrrrrr}
\toprule
 &  &  & \makecell{Event\\Count Err} & \makecell{Activity\\Dur. Error} & \makecell{Edge\\F1 Error} & \makecell{Fitness\\Error} & \makecell{Precision\\Error} & \makecell{Overall\\Error} \\
Process & Process Model & Duration Pred. &  &  &  &  &  &  \\
\midrule
\multirow[t]{9}{*}{process\_1} & combined & ml\_local & 0.000 & 0.129 & 0.385 & 0.000 & 0.000 & 0.119 \\
 & \multirow[t]{2}{*}{heuristic} & ml\_local & 0.000 & 0.129 & 0.385 & 0.000 & 0.000 & 0.119 \\
 &  & ml\_global & 0.000 & 0.178 & 0.385 & 0.000 & 0.000 & 0.127 \\
 & combined & ml\_global & 0.000 & 0.178 & 0.385 & 0.000 & 0.000 & 0.127 \\
 & heuristic & statistical & 0.000 & 0.389 & 0.385 & 0.000 & 0.000 & 0.162 \\
 & inductive & statistical & 0.000 & 0.389 & 0.385 & 0.000 & 0.000 & 0.162 \\
 & combined & statistical & 0.000 & 0.389 & 0.385 & 0.000 & 0.000